# Notebook 00 — Project Overview and Business Question

## Purpose
I use this notebook to define the project context, state the research questions,
and document the hypotheses before touching any data.

## Why this matters
Stating questions and hypotheses upfront prevents post-hoc rationalisation — a
common bias in exploratory data science.  If the question is clear before the
analysis, the results are more trustworthy.

## Inputs
None (no data required).

## Outputs
This notebook is documentation only.  No files are created.

## Connection to main question
This is the scientific foundation for all subsequent notebooks.


In [ ]:
# Standard library check — confirm the environment is set up
import sys
import importlib

REQUIRED = [
    'pandas', 'numpy', 'scipy', 'sklearn',
    'matplotlib', 'seaborn', 'plotly',
    'duckdb', 'yaml', 'shap', 'xgboost',
    'streamlit', 'joblib',
]

print(f"Python version: {sys.version}")
print()
print("Package availability check:")
all_ok = True
for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
        print(f"  OK  {pkg}")
    except ImportError:
        print(f"  MISSING  {pkg}  <-- run: pip install -r requirements.txt")
        all_ok = False

if all_ok:
    print("\nAll required packages are available.")
else:
    print("\nSome packages are missing. Run: pip install -r requirements.txt")


## Business Context

Olist is a Brazilian e-commerce marketplace aggregator.  It connects ~3,000
small sellers to customers through a single storefront on major platforms.
Orders are fulfilled by individual sellers, who are responsible for packaging
and handing items to the carrier.

**Why does this analysis matter?**

- Late deliveries are the strongest known driver of negative reviews.
- A 5-star review average improves seller ranking; poor reviews reduce it.
- Customer segmentation enables targeted retention and loyalty programmes.
- Identifying the best-performing sellers and categories guides platform investment.

## Dataset

**Olist Brazilian E-Commerce Public Dataset**
- Source: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
- Coverage: September 2016 – October 2018
- Scale: ~100,000 orders, ~99,000 unique customers, ~3,000 sellers
- Tables: 9 relational CSVs
- License: CC BY-NC-SA 4.0 (free for non-commercial research)


## Research Questions

**Primary questions:**

1. **Segmentation:** Can we use RFM (Recency, Frequency, Monetary) analysis to
   segment customers into actionable groups, and what is the revenue distribution
   across segments?

2. **Operational prediction:** Which features at order time best predict whether
   a delivery will be late?

3. **Satisfaction prediction:** Can we predict a customer's review score from
   delivery and order features, and what is the strongest predictor?

**Secondary questions:**

- Which product categories have the highest revenue and the worst delivery performance?
- Which states have the highest late-delivery rates?
- Does the repeat purchase rate differ meaningfully across customer segments?


## Hypotheses

| Hypothesis | Prediction | Falsifiable? |
|-----------|-----------|-------------|
| H1: Pareto distribution in customers | Top 20% of customers account for >60% of revenue | Yes — measure with Lorenz curve |
| H2: Delivery delay → bad reviews | Correlation between delivery_delay_days and review_score is negative and significant | Yes — Pearson/Spearman correlation + permutation test |
| H3: RF outperforms baseline | Random Forest accuracy significantly exceeds majority-class baseline on test set | Yes — bootstrap CI + permutation test |
| H4: Weight predicts late delivery | product_weight_g is in top-5 features for late delivery prediction | Yes — feature importance / SHAP |

## Alternative Explanations to Rule Out

- **Data leakage:** delivery_delay_days is known only after delivery, so it must
  NOT be used to predict whether a delivery is late (we can use it to predict review score).
- **Class imbalance:** 5-star reviews are the majority class.  A dummy model
  always predicting 5 stars may appear accurate; macro F1 is required.
- **Temporal confounding:** order volume grew over time; models trained on early
  data and tested on late data may see distribution shift.


In [ ]:
# Print a clean summary of the notebook sequence
notebooks = [
    ("00", "Project overview and hypotheses", "Documentation"),
    ("01", "Dataset selection and access", "Download guide"),
    ("02", "Data loading and first inspection", "Shapes, dtypes, .head()"),
    ("03", "Metadata and schema audit", "Column audit"),
    ("04", "Data quality checks", "Missing values, duplicates"),
    ("05", "Cleaning and feature engineering", "master.parquet"),
    ("06", "Exploratory analysis + SQL", "Charts, SQL results"),
    ("07", "RFM analysis and segmentation", "rfm_scores.parquet"),
    ("08", "Leakage control and split design", "Train/test indices"),
    ("09", "Baseline models", "Dummy + LogReg results"),
    ("10", "Advanced models", "RF + XGBoost + K-Means"),
    ("11", "Evaluation and statistical inference", "Bootstrap CIs, p-values"),
    ("12", "Robustness checks", "Ablations, shuffled labels"),
    ("13", "Interpretability and error analysis", "SHAP, confusion matrices"),
    ("14", "Publication figures and tables", "Final PNG/CSV outputs"),
    ("15", "Report generation", "Summary tables"),
    ("16", "Limitations and next steps", "Honest assessment"),
    ("17", "Dashboard preview", "Streamlit walkthrough"),
]

print(f"{'NB':>3}  {'Title':<45} {'Primary Output'}")
print("-" * 70)
for nb_id, title, output in notebooks:
    print(f"{nb_id:>3}  {title:<45} {output}")
